# Tag Extraction from Metadata

This notebook belongs to **AISTER Step 1 (Text-based metadata enrichment)** and extracts candidate tags from record metadata using rule-based heuristics and spaCy NER.

- Input: `krovets_metadata.csv` — structured metadata for the folk paintings and icons subset of the Krovets collection, generated by `01_metadata_extraction_from_europeana.ipynb`.
- Output: `krovets_tags.csv` with columns `europeana_id`, `Figures`, `Objects`, `Scenes`.

**How to run**
1. Make sure `krovets_metadata.csv` is available (either on Google Drive or in the working directory).
2. Update the `INPUT_CSV` and `OUTPUT_CSV` paths in the Configuration cell below to match your environment.
3. Run all cells from top to bottom.
4. Set `LIMIT = 0` for the full dataset, or a small value (e.g. 10) for quick testing.

## Setup

Install required dependencies.

In [1]:
!pip install -q pandas tqdm spacy
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 54.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


Import all libraries used in this notebook.

In [2]:
import os
import re
import pandas as pd
import spacy
from tqdm.auto import tqdm

try:
    from google.colab import drive, files as colab_files
except ImportError:
    drive = None
    colab_files = None

Mount Google Drive (Colab only). Skip this cell in Jupyter/local.

In [3]:
if drive is not None:
    drive.mount('/content/drive')
else:
    print('Not running in Colab; skipping Drive mount.')

Mounted at /content/drive


## Configuration

Set input/output paths and processing options. Update the paths to match your own environment.

In [4]:
# Update these paths to your own Drive/local project folder before running.
INPUT_CSV   = "/content/drive/MyDrive/W2L/Projects/AISTER/w2l/step_1/outputs/krovets_metadata.csv"
OUTPUT_CSV  = "/content/drive/MyDrive/W2L/Projects/AISTER/w2l/step_1/outputs/krovets_tags.csv"
SPACY_MODEL = "en_core_web_sm"
LIMIT       = 0  # 0 = all rows; set a small value (e.g. 10) for quick testing

## Load data

Read the metadata CSV produced by `01_metadata_extraction_from_europeana.ipynb`. In Colab, if the file is not found at the configured path, you will be prompted to upload it.

In [5]:
if not os.path.exists(INPUT_CSV):
    if colab_files is not None:
        uploaded = colab_files.upload()
        if uploaded:
            INPUT_CSV = list(uploaded.keys())[0]
        else:
            raise RuntimeError("No file uploaded.")
    else:
        raise FileNotFoundError(f"{INPUT_CSV} not found. Update INPUT_CSV to the correct path.")

df = pd.read_csv(INPUT_CSV, encoding="utf-8")
if LIMIT > 0:
    df = df.head(LIMIT).copy()
print(f"Loaded {len(df)} rows")

Loaded 312 rows


Preview the first few rows to confirm the data loaded correctly.

In [6]:
df.head(3)

,europeana_id,Title,Description,ImageLink,Creator,Subject,Type of item,Medium,Providing institution,Aggregator,Rights statement,Creation date,Places,Identifier,Is part of,Providing country,Collection name,First time published on Europeana,Last time updated from providing institution
0,HMT1,"Folk Icon ""Wedding couple""","Folk Icon ""Wedding couple"", Ukraine, Hutsulshc...",https://krovets.ua/disk/products/QCbv4ZU6ojKZH...,unknown,painting | icon | Paper | Hutsulshchyna - ethn...,painting,"wooden frame, gouache paints, paper",Online Museum of the traditional art of Ukrain...,MUSEU,http://creativecommons.org/licenses/by-nc-nd/4.0/,20th century,"Ivano-Frankivsk obl., Verkhovynskyi rn., v. Bu...",/1413/HMT1,1413_KROVETS_Museum,Ukraine,1413_KROVETS_Museum,2025-12-05T16:01:05.765Z,2025-12-05T16:01:05.765Z
1,HMT2,"Folk Icon ""St. Michael""","Folk Icon ""St. Michael"", Ukraine, Hutsulshchyn...",https://krovets.ua/disk/products/JJNPuDEex6R6W...,unknown,Wood | painting | Canvas | icon | Hutsulshchyn...,printing,"clock face, wood, levkas, canvas",Online Museum of the traditional art of Ukrain...,MUSEU,http://creativecommons.org/licenses/by-nc-nd/4.0/,20th century,"Ivano-Frankivsk obl., Verkhovynskyi rn., v. Bu...",/1413/HMT2,1413_KROVETS_Museum,Ukraine,1413_KROVETS_Museum,2025-12-05T16:01:05.765Z,2025-12-05T16:01:05.765Z
2,KYD1241,"Folk Icon ""Jesus Christ ""","Folk Icon ""Jesus Christ "", Ukraine, Western Po...",https://krovets.ua/disk/products/LMYFJXHaqKTZQ...,unknown,Western Podillia - ethnographical region | pai...,painting,"gouache paints, cardboard",Online Museum of the traditional art of Ukrain...,MUSEU,http://creativecommons.org/licenses/by-nc-nd/4.0/,20th century,"Ternopil obl., Zbarazkyi rn., v. Chornyi lis",/1413/KYD1241,1413_KROVETS_Museum,Ukraine,1413_KROVETS_Museum,2025-12-05T16:01:05.765Z,2025-12-05T16:01:05.765Z


Load the spaCy NLP model for named entity recognition.

In [7]:
NLP = spacy.load(SPACY_MODEL)
print(f"spaCy model: {SPACY_MODEL}, pipeline: {NLP.pipe_names}")

spaCy model: en_core_web_sm, pipeline: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


## Constants

Stop-word sets, lookup tables, and label sets used by the tag extraction logic.

In [8]:
# --- Media title prefixes ---
# Krovets titles use "prefix + quoted label", e.g. 'Folk Icon "St. Michael"'.
# Sorted longest-first so the most specific prefix matches first.
MEDIA_PREFIXES = sorted([
    "Professional painting", "Professional icon", "Decorative painting",
    "Folk Painting", "Folk painting", "Folk Icon", "Folk icon",
    "Old Post-card", "Old Post-Card", "Photo", "Sketch",
], key=len, reverse=True)

# --- Human-figure keywords (for Figures extraction) ---
HUMAN_ROLES = {
    "man", "woman", "boy", "girl", "child", "baby", "infant", "person",
    "mother", "father", "son", "daughter", "elder",
    "men", "women", "boys", "girls", "children", "people",
    "bride", "groom", "couple", "family", "choir",
    "saint", "angel", "apostle", "martyr",
    "kozak", "chumak", "teacher", "priest",
}

# --- Stop-word sets (tags matching these are discarded) ---
FIGURE_STOP = {
    "unknown", "group", "portrait", "group portrait", "family portrait",
    "ukraine", "photo", "photograph",
}
OBJECT_STOP = {
    "painting", "photograph", "photo", "documentary photography",
    "category:ukrainian painting", "ukraine", "ukrainian",
    "technique", "materials", "creator", "unknown",
    "century", "mid", "beginning", "half", "xx century",
    "sewing", "pottery", "printing", "formation",
    "folk", "folk icon", "folk painting", "handmade",
}
SCENE_STOP = {
    "folk painting", "folk icon", "painting", "icon", "folk",
    "photo", "photograph", "old post-card",
}

# spaCy entity labels for geographic/temporal entities (used to filter out false PERSON matches)
GEO_LABELS = {"GPE", "LOC", "FAC", "ORG", "DATE", "TIME"}

## Helper functions

In [9]:
def dedup(items):
    """Case-insensitive dedup, preserving order."""
    seen, out = set(), []
    for v in items:
        v = v.strip()
        if v and v.lower() not in seen:
            seen.add(v.lower())
            out.append(v)
    return out


def is_place(text, places):
    """True if text matches a known place name."""
    if not text or not places: return False
    low = text.strip().lower()
    return low in places or any(p in low for p in places if len(p) > 3)


def parse_media_title(title):
    """'Folk Icon "St. Michael"' -> ('Folk Icon', 'St. Michael'). Plain titles -> (None, None)."""
    for pfx in MEDIA_PREFIXES:
        if title.startswith(pfx):
            rest = title[len(pfx):].strip()
            m = re.search(r'"([^"]+)"', rest)
            return (pfx, m.group(1).strip()) if m else (pfx, rest or None)
    return None, None


def portrait_of(text):
    """'Portrait of Taras Shevchenko' -> ['Taras Shevchenko']."""
    return [m.group(1).strip().rstrip(".")
            for m in re.finditer(r"[Pp]ortrait\s+of\s+([^\"\\',\n]+)", text)
            if len(m.group(1).strip()) > 1]

## Tag extraction

The functions `extract_figures`, `extract_objects`, `extract_scenes` take a DataFrame row and return a list of candidate tags. Place names are pre-computed per row and used to filter out geographic false positives from all tag types.

### Places

Extract place names from the `Places` column and ethnographical regions in `Subject`. These are not kept as tags but are used to filter out geographic false positives from Figures, Objects, and Scenes.

In [17]:
def extract_places(row):
    """Build a set of normalised place tokens for a single row.
    Sources: Places column (split on | and ,, strip v./rn./obl. prefixes)
    and Subject column (ethnographical region names)."""
    out = set()
    for part in re.split(r"[|,]", str(row.get("Places", "") or "")):
        part = part.strip().lower()
        if not part: continue
        out.add(part)
        for px in ("v.", "rn.", "obl."):
            if part.startswith(px):
                out.add(part[len(px):].strip())
    for tok in str(row.get("Subject", "") or "").split("|"):
        m = re.match(r"(.+?)\s*-\s*ethnograph", tok.strip(), re.I)
        if m: out.add(m.group(1).strip().lower())
    out.discard("")
    return out

PLACES = {i: extract_places(row) for i, row in df.iterrows()}

In [18]:
print(PLACES[0])

{'bukovets', 'v. bukovets', 'hutsulshchyna', 'verkhovynskyi rn.', 'ivano-frankivsk obl.'}


### Figures

Depicted persons: saints (`St. Michael`), portrait subjects (`Portrait of X`), human-role keywords (`bride`, `couple`, ...), and spaCy `PERSON` entities.

In [19]:
def extract_figures(row):
    title   = str(row.get("Title", "") or "").strip()
    desc    = str(row.get("Description", "") or "").strip()
    creator = str(row.get("Creator", "") or "").strip().lower()
    places  = PLACES.get(row.name, set())
    prefix, label = parse_media_title(title)

    # For media titles use the depiction label ("St. Michael");
    # for plain titles only proceed if it contains "Portrait of ..."
    span = label or (title if portrait_of(title) else None)
    if not span:
        return []

    # 1) "Portrait of X" phrases
    figs = portrait_of(span)

    # 2) Saint names: "St. Michael", "St. George"
    figs += [f"St. {m.group(1)}"
             for m in re.finditer(r"\bSt\.\s*([A-Za-z]+(?:\s+[A-Za-z]+)?)\b", span)]

    # 3) Human-role keywords present in the label (e.g. "bride", "couple")
    words = set(re.findall(r"[a-zA-Z]+", span.lower()))
    figs += [role for role in HUMAN_ROLES if role in words]

    # 4) spaCy NER: PERSON entities from title+description,
    #    skipping any that overlap with geographic/date entities
    doc = NLP(f"{title} {desc}"[:100_000])
    geo = {(e.start_char, e.end_char) for e in doc.ents if e.label_ in GEO_LABELS}
    for ent in doc.ents:
        if ent.label_ != "PERSON": continue
        if any(ent.start_char < b and ent.end_char > a for a, b in geo): continue
        if ent.text.strip().lower() in span.lower():
            figs.append(ent.text.strip())

    # Filter out stop words, the creator name, and place names
    return dedup(f for f in figs
                 if len(f) >= 2 and f.lower() not in FIGURE_STOP
                 and f.lower() != creator and not is_place(f, places))

### Objects

Artefact names from non-media titles (e.g. `"Bowl"`, `"Woman's shirt"`), pipe-delimited Subject tokens, and spaCy `PRODUCT` entities.

In [20]:
def extract_objects(row):
    title   = str(row.get("Title", "") or "").strip()
    subject = str(row.get("Subject", "") or "").strip()
    places  = PLACES.get(row.name, set())
    prefix, _ = parse_media_title(title)

    objs = []

    # 1) Non-media titles ARE the object name (e.g. "Bowl", "Woman's shirt").
    #    Strip any quoted motif and trailing conjunctions.
    if not prefix and title:
        cleaned = re.sub(r'"[^"]*"', '', title)
        cleaned = re.sub(r'\s{2,}', ' ', cleaned).strip()
        cleaned = re.sub(r'\s+(and|or|with)\s*$', '', cleaned, flags=re.I).strip()
        if cleaned:
            objs.append(cleaned)

    # 2) Subject column: pipe-delimited tokens (skip namespace-prefixed ones like "category:...")
    for tok in subject.split("|"):
        tok = tok.strip()
        if tok and ":" not in tok:
            objs.append(tok)

    # 3) spaCy NER: PRODUCT entities from the Subject text
    if subject:
        for ent in NLP(subject[:100_000]).ents:
            if ent.label_ == "PRODUCT":
                objs.append(ent.text.strip())

    # Filter out stop words and place names
    return dedup(o for o in objs
                 if len(o) >= 2 and o.lower() not in OBJECT_STOP
                 and not is_place(o, places))

### Scenes

Depiction descriptions: the quoted label from media titles, `"Portrait of ..."` titles, and spaCy `WORK_OF_ART` / `EVENT` entities.

In [21]:
def extract_scenes(row):
    title  = str(row.get("Title", "") or "").strip()
    places = PLACES.get(row.name, set())
    prefix, label = parse_media_title(title)

    scenes = []
    if not prefix:
        # Non-media title: only use it if it's a "Portrait of ..." title
        if portrait_of(title):
            scenes.append(title)
    else:
        # Media title: the quoted depiction label is the primary scene
        if label:
            scenes.append(label)

        # Any additional quoted phrases in the title beyond the main label
        for m in re.finditer(r'"([^"]+)"', title):
            qp = m.group(1).strip()
            if qp != label:
                scenes.append(qp)

        # spaCy NER: WORK_OF_ART / EVENT entities from the label
        if label:
            for ent in NLP(label[:100_000]).ents:
                if ent.label_ in {"WORK_OF_ART", "EVENT"}:
                    t = ent.text.strip()
                    if t.lower() != label.lower():
                        scenes.append(t)

    # Filter out empty, too-long, stop-listed, metadata-prefixed, and geographic phrases
    META_PFX = ("technique", "materials", "creator")
    return dedup(s for s in scenes
                 if 2 <= len(s) <= 100 and "|" not in s
                 and s.lower() not in SCENE_STOP
                 and not any(s.lower().startswith(kw) for kw in META_PFX)
                 and not is_place(s, places))

## Run

Process every row once and extract all three tag columns.

In [79]:
fig_col, obj_col, scn_col = [], [], []

for _, row in tqdm(df.iterrows(), total=len(df), desc="Extracting tags"):
    fig_col.append(extract_figures(row))
    obj_col.append(extract_objects(row))
    scn_col.append(extract_scenes(row))

result = pd.DataFrame({
    "europeana_id": df["europeana_id"],
    "Figures": fig_col,
    "Objects": obj_col,
    "Scenes":  scn_col,
})

for col in ("Figures", "Objects", "Scenes"):
    n = result[col].apply(bool).sum()
    print(f"  {col:8s}  {n:>5d} / {len(result)}  ({100*n/len(result):.1f}%)")

Extracting tags:   0%|          | 0/312 [00:00<?, ?it/s]

  Figures     179 / 312  (57.4%)
  Objects     222 / 312  (71.2%)
  Scenes      309 / 312  (99.0%)


Preview the extracted tags for the first 10 rows.

In [27]:
result.head(10)

,europeana_id,Figures,Objects,Scenes
0,HMT1,[couple],"[icon, Paper]",[Wedding couple]
1,HMT2,[St. Michael],"[Wood, Canvas, icon]",[St. Michael]
2,KYD1241,[],"[icon, Cardboard]",[Jesus Christ]
3,KYD1242,[],"[icon, Cardboard]",[The Virgin Mary]
4,KYD1243,[],"[Wood, icon]",[The Christ]
5,KYD1244,[],"[Wood, icon]",[The Exaltation of the Holly Cross]
6,KYD1245,"[saint, Saint Michael]","[Wood, icon]",[Our Lady and Saint Michael]
7,KYD1246,[baby],"[Wood, icon]",[The Blessed Virgin with a baby]
8,KYD1247,[St. Paraskeva],"[Wood, icon]",[St. Paraskeva]
9,KYD1248,[],[],[Landscape]


## Save to CSV

Export the enriched DataFrame to CSV for use in downstream steps.

In [29]:
# Update OUTPUT_CSV at the top of this notebook if you need a different path.
result.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print(f"Saved {len(result)} rows -> {OUTPUT_CSV}")

if colab_files is not None:
    colab_files.download(OUTPUT_CSV)

Saved 312 rows -> /content/drive/MyDrive/W2L/Projects/AISTER/w2l/step_1/outputs/krovets_tags.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>